<a href="https://colab.research.google.com/github/arulbenjaminchandru/ai-architect-program/blob/main/Day_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
## Section 1 - What is Tool Use?

### 🧠 The Big Idea

Imagine you are a very smart manager at a company.  
You know a LOT — strategy, history, language, logic.  
But you do NOT personally do every task yourself.  
When someone asks "what time is it in Tokyo?", you pick up your phone and check.  
When someone asks you to calculate a 200-row spreadsheet, you open Excel.

**You are the brain. The tools are your specialists.**

Claude works exactly the same way:
- Claude is brilliant at language, reasoning, and knowledge.
- But Claude cannot look up today's stock price, run Python code, or check live weather.
- So we give Claude **tools** — and Claude decides when to use them.

---

### 📱 Analogy 2 — The Calculator on Your Phone

When you open your phone's calculator, YOUR BRAIN decides:
- "This math is too complex to do in my head — I'll open the calculator."
- The calculator just does the arithmetic. You decide WHEN to use it.

Claude = Your Brain  
Tools = Specialist Apps (Calculator, Clock, Search Engine)

---

### 🔧 What Exactly IS a Tool?

A **tool** has two parts:

| Part | What it is | Example |
|------|-----------|---------|
| **Python function** | The actual code that runs | `def get_current_time(): ...` |
| **JSON schema** | A description Claude can read | `{"name": "get_current_time", ...}` |

The **JSON schema** is like a job description for the tool.  
Claude reads it to understand: "What is this tool called? What does it do? What inputs does it need?"

---

### 👀 Example: The Simplest Possible Tool

Here is a tool that requires **no inputs** at all — it just returns the current time.  
Notice: the schema is a Python dictionary that describes the function to Claude.


In [1]:
import datetime

# --- THE PYTHON FUNCTION ---
# This is the actual code that will run when the tool is called.
def get_current_time():
    current_time = datetime.datetime.now()
    formatted_time = current_time.strftime("%Y-%m-%d %H:%M:%S")
    return formatted_time

# --- THE JSON SCHEMA (Tool Definition) ---
# This is what we show to Claude so it understands the tool.
# Claude reads this description — it never sees the Python function directly.
get_current_time_tool = {
    "name": "get_current_time",
    "description": "Returns the current date and time. Use this when the user asks what time or date it is right now.",
    "input_schema": {
        "type": "object",
        "properties": {},       # No inputs needed for this tool
        "required": []          # Nothing is required
    }
}

# Let's test the function directly (without Claude)
result = get_current_time()
print(f"Current time: {result}")
print()
print("Tool schema (what Claude will see):")
print(get_current_time_tool)


Current time: 2026-05-02 06:26:32

Tool schema (what Claude will see):
{'name': 'get_current_time', 'description': 'Returns the current date and time. Use this when the user asks what time or date it is right now.', 'input_schema': {'type': 'object', 'properties': {}, 'required': []}}


---

### 📋 When Does the AI Use a Tool?

Not every question needs a tool. Claude is smart — it only calls a tool when it genuinely needs one.

| User Prompt | Does AI use a tool? | Why |
|---|---|---|
| "What is 2 + 2?" | ❌ No | AI knows this directly from training |
| "What time is it in Tokyo right now?" | ✅ Yes | Needs live data — AI's training is not real-time |
| "Explain what photosynthesis is" | ❌ No | General knowledge from training |
| "What is the BMI of someone who is 170cm and 70kg?" | ✅ Yes | A calculation tool makes this precise and reliable |

---

### 🧠 Quick Check — Think Before You Run!

**Look at these two prompts. Which one do you think needs a tool, and why?**

**(a)** "Who wrote Romeo and Juliet?"  
**(b)** "What is today's date?"

💬 Discuss with the person next to you for 60 seconds, then we'll share answers as a class.

> **Answer:** (a) No tool needed — Claude knows Shakespeare from training.  
> (b) Tool needed — Claude was trained on data from the past and cannot know today's date without a real-time tool.


---
## Section 2 — How the AI Decides to Call a Tool

### 🔍 Claude's Decision Process

When you send a message to Claude with tools available, Claude follows this thought process:

1. **Read the user's message** — What is being asked?
2. **Read the tool descriptions** — What tools are available?
3. **Decide:** "Can I answer this from my training data? Or do I need a tool?"
4. **If no tool needed:** Claude replies with a text answer directly.
5. **If a tool is needed:** Claude returns a `tool_use` block — it does NOT run the tool itself.

🔑 **The most important insight of this section:**  
**Claude NEVER runs your Python function. YOU do.**  
Claude just says: "Please call this function with these inputs."  
Your code is responsible for actually running the function and sending the result back.

---

### 📝 Example A — AI Answers Directly (No Tool Needed)

```
User: "What does BMI stand for?"

AI Response:
  "BMI stands for Body Mass Index. It is a numerical value derived from
   a person's weight and height, used as a screening tool to categorize
   whether a person has a healthy body weight."

→ No tool_use block. Claude answered from training knowledge.
  Even if you had a calculate_bmi tool available, Claude didn't need it here.
```

---

### 📝 Example B — AI Requests a Tool Call

```
User: "What is the BMI of someone who is 170cm tall and weighs 68kg?"

AI Raw Response (simplified):
{
  "type": "tool_use",
  "name": "calculate_bmi",
  "input": {
    "height_cm": 170,
    "weight_kg": 68
  }
}

→ Claude did NOT calculate the BMI itself.
  It identified that calculate_bmi is the right tool.
  It figured out the correct inputs (170, 68) from the user's message.
  Now YOUR CODE must run calculate_bmi(170, 68) and send the result back.
```

---

### 📝 Example C — Ambiguous Prompt (Great Discussion Point!)

```
User: "Is 25 a good BMI?"

→ Interesting case! Claude might:
  (a) Answer from training: "A BMI of 25 is at the upper edge of the 'Normal' range..."
  (b) Call a tool — IF you defined a tool like check_bmi_category(bmi_value)

This shows a key lesson: Tool use depends on WHAT TOOLS YOU OFFER.
If you define a tool for it, Claude is more likely to use the tool for precision.
If no tool is defined, Claude falls back to its training knowledge.
```

---

### 📝 Example D — Wrong Tools Offered (Showing Limitations)

```
User: "What is the weather in Chennai right now?"

Tools available: [calculate_bmi, convert_currency]

→ Claude will politely say it cannot help with weather information.
  It has no weather tool. It cannot make up data.
  
Lesson: Claude can ONLY use tools YOU define. It cannot invent new tools.
        If you need weather data, you must define and provide a weather tool.
```

---

### 🧠 Quick Check

**Scenario:** You give Claude a tool called `search_web(query)`.  
You ask: *"Who won the IPL 2024?"*

**What do you expect to happen?**

> **Answer:** Claude will return a `tool_use` block asking your code to call  
> `search_web(query="IPL 2024 winner")`. Your code runs the search,  
> gets the result, and sends it back. Claude then tells the user the answer.


In [3]:
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 11.6 MB/s eta 0:00:00


In [4]:
from anthropic import Anthropic
from google.colab import userdata
key = userdata.get('MY_API_KEY')

client = Anthropic(api_key=key)

In [5]:
# Let's see the difference between a direct answer and a tool_use response
# We'll send TWO different prompts and observe Claude's behavior

# First, define a simple BMI tool so Claude has it available
calculate_bmi_tool = {
    "name": "calculate_bmi",
    "description": "Calculates the Body Mass Index (BMI) for a person given their height and weight.",
    "input_schema": {
        "type": "object",
        "properties": {
            "height_cm": {
                "type": "number",
                "description": "The person's height in centimetres"
            },
            "weight_kg": {
                "type": "number",
                "description": "The person's weight in kilograms"
            }
        },
        "required": ["height_cm", "weight_kg"]
    }
}

# --- PROMPT A: No tool needed ---
print("=== PROMPT A: Question Claude can answer from knowledge ===")
response_a = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=300,
    tools=[calculate_bmi_tool],
    messages=[
        {"role": "user", "content": "What does BMI stand for?"}
    ]
)

print(f"Stop reason: {response_a.stop_reason}")
print(f"Number of content blocks: {len(response_a.content)}")
for content_block in response_a.content:
    print(f"Content type: {content_block.type}")
    if content_block.type == "text":
        print(f"Text: {content_block.text}")

print()
print("=== PROMPT B: Question that needs the tool ===")
response_b = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=300,
    tools=[calculate_bmi_tool],
    messages=[
        {"role": "user", "content": "What is the BMI of someone who is 170cm tall and weighs 68kg?"}
    ]
)

print(f"Stop reason: {response_b.stop_reason}")
print(f"Number of content blocks: {len(response_b.content)}")
for content_block in response_b.content:
    print(f"Content type: {content_block.type}")
    if content_block.type == "tool_use":
        print(f"Tool name: {content_block.name}")
        print(f"Tool inputs: {content_block.input}")
        print(f"Tool use ID: {content_block.id}")


=== PROMPT A: Question Claude can answer from knowledge ===
Stop reason: end_turn
Number of content blocks: 1
Content type: text
Text: BMI stands for **Body Mass Index**.

It's a measure of body fat based on height and weight that applies to adult men and women. BMI is calculated by dividing a person's weight in kilograms by the square of their height in meters (kg/m²).

BMI is commonly used as a screening tool to identify potential weight categories that may lead to health problems, though it's important to note that it doesn't directly measure body fat percentage and has some limitations, particularly for athletes, elderly individuals, or those with significant muscle mass.

If you'd like, I can calculate the BMI for you if you provide your height and weight!

=== PROMPT B: Question that needs the tool ===
Stop reason: tool_use
Number of content blocks: 1
Content type: tool_use
Tool name: calculate_bmi
Tool inputs: {'height_cm': 170, 'weight_kg': 68}
Tool use ID: toolu_01JfobQo3sBDQP

---
## Section 3 — The Tool Call Loop (Core Coding Block)

### 🔄 The 5-Step Loop

Every tool call in Claude follows this exact same pattern — no exceptions.  
Memorise this loop and you understand 90% of tool use.

```
Step 1: You send a message + tool definitions to Claude
        ↓
Step 2: Claude returns a tool_use block (NOT an answer yet!)
        ↓
Step 3: Your code detects tool_use and runs the local Python function
        ↓
Step 4: You send the tool result back to Claude
        ↓
Step 5: Claude reads the result and gives the final answer to the user
```

Think of it like a relay race:
- You pass the baton to Claude (Step 1)
- Claude passes back a request (Step 2)
- Your code runs the function (Step 3)
- Your code passes the result to Claude (Step 4)
- Claude finishes the race with a proper answer (Step 5)

---

### Example A — Single Tool, Single Call: BMI Calculator


In [6]:
# ============================================================
# EXAMPLE A: Single tool, single call — BMI Calculator
# ============================================================
# This is the complete tool call loop, fully commented.
# Read EVERY comment — they explain what is happening and why.
# ============================================================

# --- STEP 0: Define the tool and the local Python function ---

# The Python function — this is what ACTUALLY calculates the BMI
def calculate_bmi(height_cm, weight_kg):
    # BMI formula: weight(kg) / (height(m))^2
    height_in_metres = height_cm / 100
    bmi_value = weight_kg / (height_in_metres * height_in_metres)
    # Round to 2 decimal places for a clean result
    rounded_bmi = round(bmi_value, 2)
    return rounded_bmi

# The tool schema — this is what Claude reads to understand the tool
bmi_tool_definition = {
    "name": "calculate_bmi",
    "description": "Calculates Body Mass Index from height and weight.",
    "input_schema": {
        "type": "object",
        "properties": {
            "height_cm": {
                "type": "number",
                "description": "Height in centimetres, e.g. 170"
            },
            "weight_kg": {
                "type": "number",
                "description": "Weight in kilograms, e.g. 68"
            }
        },
        "required": ["height_cm", "weight_kg"]
    }
}

# The user's question
user_question = "What is the BMI of someone who is 170cm tall and weighs 68kg?"

print("--- Step 1: Sending the user's message + tool definitions to Claude ---")
print(f"User asked: {user_question}")

# Send the message to Claude, including our tool definition
first_response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=500,
    tools=[bmi_tool_definition],          # We offer this tool to Claude
    messages=[
        {"role": "user", "content": user_question}
    ]
)

print()
print(f"--- Step 2: Claude responded. Stop reason: {first_response.stop_reason} ---")
# stop_reason == "tool_use" means Claude wants us to run a tool
# stop_reason == "end_turn" means Claude gave a direct answer (no tool needed)

# We need to store Claude's full response to send back in the next turn
# This is required — Claude needs to "remember" what it said
claude_first_response_content = first_response.content

# Look through Claude's response to find the tool_use block
tool_use_block = None
for content_block in first_response.content:
    if content_block.type == "tool_use":
        tool_use_block = content_block
        print(f"Claude wants to call tool: {tool_use_block.name}")
        print(f"With inputs: {tool_use_block.input}")
        print(f"Tool use ID: {tool_use_block.id}")

print()
print("--- Step 3: Our code runs the local Python function ---")
# Extract the inputs Claude requested
height_input = tool_use_block.input["height_cm"]
weight_input = tool_use_block.input["weight_kg"]

# Run our local function — Claude does NOT do this. WE do.
calculated_bmi_result = calculate_bmi(height_input, weight_input)
print(f"Our function returned: BMI = {calculated_bmi_result}")

print()
print("--- Step 4: Sending the tool result back to Claude ---")
# We must construct the messages list carefully:
# [original user message, Claude's tool_use response, our tool result]
messages_with_tool_result = [
    # The original user question
    {"role": "user", "content": user_question},
    # Claude's response that contained the tool_use request
    {"role": "assistant", "content": claude_first_response_content},
    # Our tool result — note the special structure required
    {
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": tool_use_block.id,    # Must match Claude's request ID
                "content": str(calculated_bmi_result)  # The actual result as a string
            }
        ]
    }
]

# Send everything back to Claude
final_response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=500,
    tools=[bmi_tool_definition],
    messages=messages_with_tool_result
)

print()
print("--- Step 5: Claude's final answer to the user ---")
for content_block in final_response.content:
    if content_block.type == "text":
        print(content_block.text)


--- Step 1: Sending the user's message + tool definitions to Claude ---
User asked: What is the BMI of someone who is 170cm tall and weighs 68kg?

--- Step 2: Claude responded. Stop reason: tool_use ---
Claude wants to call tool: calculate_bmi
With inputs: {'height_cm': 170, 'weight_kg': 68}
Tool use ID: toolu_012fP72bANqwqUwnW4yrxv6z

--- Step 3: Our code runs the local Python function ---
Our function returned: BMI = 23.53

--- Step 4: Sending the tool result back to Claude ---

--- Step 5: Claude's final answer to the user ---
The BMI of someone who is 170cm tall and weighs 68kg is **23.53**.

This falls within the "normal weight" category, as a BMI between 18.5 and 24.9 is generally considered healthy.


---

### Example B — Single Tool, Edge Case: Temperature Converter

This example highlights something important:  
Claude does not just relay the tool result — it **reasons** about it.  
Watch how Claude adds medical context after receiving the temperature conversion.


In [7]:
# ============================================================
# EXAMPLE B: Temperature converter — AI reasons about the result
# ============================================================

# The Python function for temperature conversion
def convert_celsius_to_fahrenheit(celsius):
    fahrenheit_value = (celsius * 9 / 5) + 32
    rounded_result = round(fahrenheit_value, 1)
    return rounded_result

# The tool schema
temperature_tool_definition = {
    "name": "convert_celsius_to_fahrenheit",
    "description": "Converts a temperature from Celsius to Fahrenheit.",
    "input_schema": {
        "type": "object",
        "properties": {
            "celsius": {
                "type": "number",
                "description": "Temperature in Celsius, e.g. 38.5"
            }
        },
        "required": ["celsius"]
    }
}

# This prompt is interesting — the user is asking TWO things:
# 1. Convert 38.5°C to Fahrenheit (needs tool)
# 2. Is it a fever? (Claude can reason about this from knowledge)
temperature_question = "My body temperature is 38.5°C. What is that in Fahrenheit? Is it a fever?"

print("--- Step 1: Sending question to Claude ---")
print(f"User asked: {temperature_question}")

first_response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=500,
    tools=[temperature_tool_definition],
    messages=[
        {"role": "user", "content": temperature_question}
    ]
)

print(f"Stop reason: {first_response.stop_reason}")

# Find and run the tool
tool_use_block = None
for content_block in first_response.content:
    if content_block.type == "tool_use":
        tool_use_block = content_block

print()
print(f"--- Step 2 & 3: Claude requested tool '{tool_use_block.name}' ---")
celsius_input = tool_use_block.input["celsius"]
fahrenheit_result = convert_celsius_to_fahrenheit(celsius_input)
print(f"Converted {celsius_input}°C → {fahrenheit_result}°F")

print()
print("--- Step 4: Sending result back to Claude ---")
messages_for_second_call = [
    {"role": "user", "content": temperature_question},
    {"role": "assistant", "content": first_response.content},
    {
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": tool_use_block.id,
                "content": str(fahrenheit_result)
            }
        ]
    }
]

final_response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=500,
    tools=[temperature_tool_definition],
    messages=messages_for_second_call
)

print()
print("--- Step 5: Claude's final answer (notice it adds medical reasoning!) ---")
for content_block in final_response.content:
    if content_block.type == "text":
        print(content_block.text)

print()
print("👆 Notice: Claude used the tool result AND added its own medical knowledge.")
print("   The tool gave the number. Claude gave the interpretation.")


--- Step 1: Sending question to Claude ---
User asked: My body temperature is 38.5°C. What is that in Fahrenheit? Is it a fever?
Stop reason: tool_use

--- Step 2 & 3: Claude requested tool 'convert_celsius_to_fahrenheit' ---
Converted 38.5°C → 101.3°F

--- Step 4: Sending result back to Claude ---

--- Step 5: Claude's final answer (notice it adds medical reasoning!) ---
Your body temperature of 38.5°C is **101.3°F**.

**Yes, this is a fever.** Here's how it compares to normal:

- **Normal body temperature**: 36.5-37.5°C (97.7-99.5°F)
- **Low-grade fever**: 37.5-38.5°C (99.5-101.3°F)
- **Moderate fever**: 38.5-39.5°C (101.3-103.1°F)
- **High fever**: Above 39.5°C (103.1°F)

At 38.5°C, you're at the borderline between a low-grade and moderate fever. While fevers are often a sign that your body is fighting an infection, a temperature at this level is generally not considered dangerous in adults. However, you should:

- Stay hydrated
- Rest
- Monitor your symptoms
- Consult a healthcare 

---

### Example C — Tool Called Twice in One Session

Sometimes Claude needs to call the **same tool multiple times** in a single response.  
For example: "Compare the BMI of Person A and Person B."  
Claude will request TWO separate tool calls — one for each person.

Your code must loop through ALL content blocks and handle each one.


In [8]:
# ============================================================
# EXAMPLE C: Handling multiple tool calls in a single response
# ============================================================

# Reuse the calculate_bmi function from Example A

comparison_question = (
    "Compare the BMI of Person A (160cm, 55kg) and Person B (180cm, 90kg). "
    "Who is healthier by BMI standards?"
)

print("--- Step 1: Sending the comparison question to Claude ---")
print(f"User asked: {comparison_question}")

first_response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1000,
    tools=[bmi_tool_definition],
    messages=[
        {"role": "user", "content": comparison_question}
    ]
)

print(f"Stop reason: {first_response.stop_reason}")
print(f"Number of content blocks in response: {len(first_response.content)}")

print()
print("--- Step 2: Inspecting all content blocks ---")
for block in first_response.content:
    print(f"  Block type: {block.type}")
    if block.type == "tool_use":
        print(f"    Tool: {block.name}, Inputs: {block.input}")

print()
print("--- Step 3: Running ALL tool calls that Claude requested ---")

# This list will collect all tool results
all_tool_results = []

# Loop through every content block and handle each tool_use
for content_block in first_response.content:
    if content_block.type == "tool_use":
        # Extract inputs for this specific tool call
        height_input = content_block.input["height_cm"]
        weight_input = content_block.input["weight_kg"]

        # Run the local function
        bmi_result = calculate_bmi(height_input, weight_input)
        print(f"  Calculated BMI for {height_input}cm / {weight_input}kg = {bmi_result}")

        # Build the tool result entry for this call
        single_tool_result = {
            "type": "tool_result",
            "tool_use_id": content_block.id,   # Match the exact ID from Claude's request
            "content": str(bmi_result)
        }
        all_tool_results.append(single_tool_result)

print()
print("--- Step 4: Sending all tool results back to Claude in one message ---")
messages_with_all_results = [
    {"role": "user", "content": comparison_question},
    {"role": "assistant", "content": first_response.content},
    {
        "role": "user",
        "content": all_tool_results    # Send ALL results at once
    }
]

final_response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1000,
    tools=[bmi_tool_definition],
    messages=messages_with_all_results
)

print()
print("--- Step 5: Claude's final comparison answer ---")
for content_block in final_response.content:
    if content_block.type == "text":
        print(content_block.text)


--- Step 1: Sending the comparison question to Claude ---
User asked: Compare the BMI of Person A (160cm, 55kg) and Person B (180cm, 90kg). Who is healthier by BMI standards?
Stop reason: tool_use
Number of content blocks in response: 3

--- Step 2: Inspecting all content blocks ---
  Block type: text
  Block type: tool_use
    Tool: calculate_bmi, Inputs: {'height_cm': 160, 'weight_kg': 55}
  Block type: tool_use
    Tool: calculate_bmi, Inputs: {'height_cm': 180, 'weight_kg': 90}

--- Step 3: Running ALL tool calls that Claude requested ---
  Calculated BMI for 160cm / 55kg = 21.48
  Calculated BMI for 180cm / 90kg = 27.78

--- Step 4: Sending all tool results back to Claude in one message ---

--- Step 5: Claude's final comparison answer ---
Based on the BMI calculations:

**Person A:** BMI = 21.48
**Person B:** BMI = 27.78

**Person A is healthier by BMI standards.**

Here's the breakdown:
- **Person A's BMI (21.48)** falls into the **"Normal Weight"** category (BMI 18.5-24.9), whi

---

### 🧠 Quick Check — Draw the Flow!

**Question:**  
In Example C above:
- How many times does your code call Claude (the API)?
- How many times does it call the local `calculate_bmi` Python function?
- Draw the 5-step loop for this specific example.

> **Answer:**  
> - Claude API calls: **2** (once to get the tool requests, once to get the final answer)  
> - Local function calls: **2** (once for Person A, once for Person B)  
> - The loop: Send → Detect 2 tool_use blocks → Run function twice → Return both results → Final answer


---
## Section 4 — Multiple Tools & Real Patterns

### 🧰 Offering Claude a Toolbox

So far, we gave Claude one tool at a time.  
In real applications, you offer Claude a **set of tools** and Claude picks the right one.

This is powerful because:
- You build the tools once
- Claude figures out which tool to use for each user request
- You don't have to hard-code "if the user asks about BMI, use this tool"

The pattern is called a **tool registry** — a dictionary that maps tool names to Python functions.

---

### 📝 All Example Prompts for This Section

Before we run code, let's preview how tool selection will work:

**Prompt 1 → Tool: `calculate_bmi`**
```
"A person is 165cm and weighs 72kg. What is their BMI?"
→ Claude picks: calculate_bmi(height_cm=165, weight_kg=72)
```

**Prompt 2 → Tool: `convert_currency`**
```
"How much is 5000 Indian Rupees in US Dollars?"
→ Claude picks: convert_currency(amount=5000, from_currency="INR", to_currency="USD")
```

**Prompt 3 → Tool: `get_word_count`**
```
"How many words are in: The quick brown fox jumps over the lazy dog"
→ Claude picks: get_word_count(text="The quick brown fox jumps over the lazy dog")
```

**Prompt 4 → Two Tools at Once**
```
"I weigh 80kg and am 175cm. Also convert my monthly salary of ₹85,000 to USD."
→ Claude picks: calculate_bmi AND convert_currency
```

**Prompt 5 → No Tool**
```
"What are the health risks of a BMI over 30?"
→ Claude answers from knowledge. No tool called.
  Having tools available does NOT force Claude to use them.
```


In [9]:
# ============================================================
# SECTION 4: Multiple tools — Claude picks the right one
# ============================================================

# --- Define all three Python functions ---

def calculate_bmi(height_cm, weight_kg):
    height_in_metres = height_cm / 100
    bmi_value = weight_kg / (height_in_metres * height_in_metres)
    return round(bmi_value, 2)

def convert_currency(amount, from_currency, to_currency):
    # Mock exchange rates — in a real app, you'd call a live API
    mock_rates = {
        ("INR", "USD"): 0.012,
        ("USD", "INR"): 83.5,
        ("INR", "EUR"): 0.011,
        ("USD", "EUR"): 0.92,
        ("EUR", "USD"): 1.09
    }
    rate_key = (from_currency.upper(), to_currency.upper())
    if rate_key in mock_rates:
        converted_amount = amount * mock_rates[rate_key]
        return round(converted_amount, 2)
    else:
        return f"Rate not available for {from_currency} to {to_currency}"

def get_word_count(text):
    words_list = text.split()
    word_count = len(words_list)
    return word_count

# --- Define all three tool schemas ---

bmi_tool = {
    "name": "calculate_bmi",
    "description": "Calculates the Body Mass Index from height in cm and weight in kg.",
    "input_schema": {
        "type": "object",
        "properties": {
            "height_cm": {"type": "number", "description": "Height in centimetres"},
            "weight_kg": {"type": "number", "description": "Weight in kilograms"}
        },
        "required": ["height_cm", "weight_kg"]
    }
}

currency_tool = {
    "name": "convert_currency",
    "description": "Converts an amount from one currency to another. Supports INR, USD, EUR.",
    "input_schema": {
        "type": "object",
        "properties": {
            "amount": {"type": "number", "description": "The amount to convert"},
            "from_currency": {"type": "string", "description": "Source currency code, e.g. INR"},
            "to_currency": {"type": "string", "description": "Target currency code, e.g. USD"}
        },
        "required": ["amount", "from_currency", "to_currency"]
    }
}

word_count_tool = {
    "name": "get_word_count",
    "description": "Counts the number of words in a given piece of text.",
    "input_schema": {
        "type": "object",
        "properties": {
            "text": {"type": "string", "description": "The text to count words in"}
        },
        "required": ["text"]
    }
}

# --- The tool registry: maps tool names to Python functions ---
# When Claude says "call calculate_bmi", we look up this dict to find the function.
tool_registry = {
    "calculate_bmi": calculate_bmi,
    "convert_currency": convert_currency,
    "get_word_count": get_word_count
}

# All tools in one list — we pass this to every Claude call
all_tools = [bmi_tool, currency_tool, word_count_tool]

print("✅ Three tools defined and registered.")
print(f"Tool registry keys: {list(tool_registry.keys())}")


✅ Three tools defined and registered.
Tool registry keys: ['calculate_bmi', 'convert_currency', 'get_word_count']


Now let's build a **reusable tool loop function** — so we don't repeat the same code for every prompt.  
This is how real applications work: one generic loop handles all tools.


In [10]:
# ============================================================
# A reusable function that handles the full tool call loop
# ============================================================

def run_tool_call_loop(user_message, available_tools, tool_registry):
    # Runs the complete tool call loop for a given user message.
    # Handles any number of tool calls in a single response.
    # Returns the final text answer from Claude.
    print(f"User: {user_message}")
    print()

    # Build the initial messages list
    messages = [
        {"role": "user", "content": user_message}
    ]

    # Keep looping until Claude gives a final text answer
    loop_count = 0
    while True:
        loop_count = loop_count + 1
        print(f"--- API Call #{loop_count} ---")

        # Send to Claude
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1000,
            tools=available_tools,
            messages=messages
        )

        print(f"Stop reason: {response.stop_reason}")

        # If Claude is done with tools, extract the final text answer
        if response.stop_reason == "end_turn":
            final_answer = ""
            for content_block in response.content:
                if content_block.type == "text":
                    final_answer = final_answer + content_block.text
            return final_answer

        # Otherwise, process all tool calls in the response
        all_tool_results = []
        for content_block in response.content:
            if content_block.type == "tool_use":
                tool_name = content_block.name
                tool_inputs = content_block.input
                tool_id = content_block.id

                print(f"  Claude requested: {tool_name}({tool_inputs})")

                # Look up the function in our registry and call it
                python_function = tool_registry[tool_name]
                tool_output = python_function(**tool_inputs)

                print(f"  Our function returned: {tool_output}")

                # Collect this result
                single_result = {
                    "type": "tool_result",
                    "tool_use_id": tool_id,
                    "content": str(tool_output)
                }
                all_tool_results.append(single_result)

        # Add Claude's response and our tool results to the conversation
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": all_tool_results})

print("✅ Tool loop function defined. Ready to test with all 5 prompts!")


✅ Tool loop function defined. Ready to test with all 5 prompts!


In [11]:
# ============================================================
# Test Prompt 1: Routes to calculate_bmi
# ============================================================
print("=" * 60)
print("PROMPT 1: BMI Calculation")
print("=" * 60)
answer_1 = run_tool_call_loop(
    "A person is 165cm and weighs 72kg. What is their BMI?",
    all_tools,
    tool_registry
)
print()
print("Claude's Answer:")
print(answer_1)


PROMPT 1: BMI Calculation
User: A person is 165cm and weighs 72kg. What is their BMI?

--- API Call #1 ---
Stop reason: tool_use
  Claude requested: calculate_bmi({'height_cm': 165, 'weight_kg': 72})
  Our function returned: 26.45
--- API Call #2 ---
Stop reason: end_turn

Claude's Answer:
The BMI of a person who is 165 cm tall and weighs 72 kg is **26.45**.

This falls into the **overweight** category according to standard BMI classifications:
- Underweight: BMI < 18.5
- Normal weight: BMI 18.5 - 24.9
- Overweight: BMI 25.0 - 29.9
- Obese: BMI ≥ 30.0


In [12]:
# ============================================================
# Test Prompt 2: Routes to convert_currency
# ============================================================
print("=" * 60)
print("PROMPT 2: Currency Conversion")
print("=" * 60)
answer_2 = run_tool_call_loop(
    "How much is 5000 Indian Rupees in US Dollars?",
    all_tools,
    tool_registry
)
print()
print("Claude's Answer:")
print(answer_2)


PROMPT 2: Currency Conversion
User: How much is 5000 Indian Rupees in US Dollars?

--- API Call #1 ---
Stop reason: tool_use
  Claude requested: convert_currency({'amount': 5000, 'from_currency': 'INR', 'to_currency': 'USD'})
  Our function returned: 60.0
--- API Call #2 ---
Stop reason: end_turn

Claude's Answer:
5000 Indian Rupees is equal to **$60.00 USD**.


In [13]:
# ============================================================
# Test Prompt 3: Routes to get_word_count
# ============================================================
print("=" * 60)
print("PROMPT 3: Word Count")
print("=" * 60)
answer_3 = run_tool_call_loop(
    "How many words are in this sentence: The quick brown fox jumps over the lazy dog",
    all_tools,
    tool_registry
)
print()
print("Claude's Answer:")
print(answer_3)


PROMPT 3: Word Count
User: How many words are in this sentence: The quick brown fox jumps over the lazy dog

--- API Call #1 ---
Stop reason: tool_use
  Claude requested: get_word_count({'text': 'The quick brown fox jumps over the lazy dog'})
  Our function returned: 9
--- API Call #2 ---
Stop reason: end_turn

Claude's Answer:
The sentence "The quick brown fox jumps over the lazy dog" contains **9 words**.


In [14]:
# ============================================================
# Test Prompt 4: Requires TWO different tools
# ============================================================
print("=" * 60)
print("PROMPT 4: Two Tools in One Request")
print("=" * 60)
answer_4 = run_tool_call_loop(
    "I weigh 80kg and am 175cm. Also convert my monthly salary of 85000 INR to USD. Give me both answers.",
    all_tools,
    tool_registry
)
print()
print("Claude's Answer:")
print(answer_4)


PROMPT 4: Two Tools in One Request
User: I weigh 80kg and am 175cm. Also convert my monthly salary of 85000 INR to USD. Give me both answers.

--- API Call #1 ---
Stop reason: tool_use
  Claude requested: calculate_bmi({'height_cm': 175, 'weight_kg': 80})
  Our function returned: 26.12
  Claude requested: convert_currency({'amount': 85000, 'from_currency': 'INR', 'to_currency': 'USD'})
  Our function returned: 1020.0
--- API Call #2 ---
Stop reason: end_turn

Claude's Answer:
Great! Here are both answers:

**BMI:** Your Body Mass Index is **26.12**, which falls into the "overweight" category (normal BMI is 18.5-24.9).

**Salary Conversion:** Your monthly salary of 85,000 INR converts to **$1,020 USD**.


In [15]:
# ============================================================
# Test Prompt 5: No tool needed — Claude answers from knowledge
# ============================================================
print("=" * 60)
print("PROMPT 5: No Tool Needed")
print("=" * 60)
answer_5 = run_tool_call_loop(
    "What are the health risks of a BMI over 30?",
    all_tools,
    tool_registry
)
print()
print("Claude's Answer:")
print(answer_5)
print()
print("👆 Notice: stop_reason was 'end_turn' immediately — no tool was called!")
print("   Having tools available does NOT force Claude to use them.")


PROMPT 5: No Tool Needed
User: What are the health risks of a BMI over 30?

--- API Call #1 ---
Stop reason: end_turn

Claude's Answer:
A BMI over 30 is classified as **obese**, and there are numerous health risks associated with this range:

## Major Health Risks of BMI > 30:

**Cardiovascular Diseases:**
- Increased risk of heart disease and heart attack
- High blood pressure (hypertension)
- Stroke
- Abnormal cholesterol levels

**Metabolic Disorders:**
- Type 2 diabetes
- Insulin resistance
- Metabolic syndrome

**Joint and Bone Problems:**
- Osteoarthritis, especially in weight-bearing joints (knees, hips, lower back)
- Increased injury risk
- Reduced mobility and flexibility

**Respiratory Issues:**
- Sleep apnea
- Obesity hypoventilation syndrome
- Increased asthma risk

**Cancer Risk:**
- Increased risk of several cancers including breast, colon, endometrial, and prostate cancer

**Digestive Problems:**
- Gastroesophageal reflux disease (GERD)
- Fatty liver disease
- Gallstones

---

### 🧠 Quick Check — Design Your Own Toolbox!

**You are building a fitness app. List 3 tools you would define.**  
For each tool, write:
1. The tool name
2. What it does in one sentence  
3. A sample user prompt that would trigger it

**Example answer:**

| Tool Name | What it does | Sample Prompt |
|---|---|---|
| `calculate_calories_burned` | Estimates calories burned during exercise | "I ran for 30 minutes at moderate pace. How many calories did I burn?" |
| `get_water_intake_goal` | Returns daily water intake recommendation by weight | "I weigh 70kg. How much water should I drink per day?" |
| `check_heart_rate_zone` | Classifies a heart rate into training zones | "My heart rate is 145bpm. What training zone is that?" |

💬 Write your own 3 tools in the cell below or discuss with your neighbour.


---
## Section 5 — Wrap-Up

---

## 🎉 What You Learned Today

Congratulations on completing the Tool Use session! Here's a summary of everything you now know:

---

✅ **A tool is a Python function + a JSON schema that describes it to Claude**  
Claude reads the schema to understand what the tool does and what inputs it needs.  
Your Python function is the actual code that runs — Claude never sees it directly.

---

✅ **Claude decides WHETHER to call a tool — your code decides HOW to run it**  
Claude never executes your code. It only requests tool calls.  
You are always in control of execution.

---

✅ **The tool call loop is always: Send → Detect → Execute → Return → Final Answer**  
Every tool use follows this exact 5-step pattern, no exceptions.  
Once you understand this loop, you can build any tool-powered application.

---

✅ **You can offer multiple tools — Claude picks the right one automatically**  
Build a tool registry. Pass all tools to Claude. Claude selects based on the user's intent.  
You never have to hard-code "if user asks X, use tool Y."

---

✅ **Having tools available does NOT force Claude to use them**  
Claude is intelligent. If it can answer from training data, it will.  
Tools are used only when they genuinely add value.

---

## 📚 Continue Learning

- **Anthropic Tool Use Documentation:** https://docs.anthropic.com/en/docs/tool-use  
---

*Thank you for attending! Questions? Reach out via the WhatsApp group*
